# What Characteristics Influence Decision Outcomes?

---

**Task:** Investigate what factors such as institution, ROTC status, major, GPA, and demographics, predict selection into the VICEROY program, using logistic GEE to account for institutional clustering.

VICEROY does not select from a unified applicant pool. Each partner university independently nominates its own applicants, which means:

- A student's odds depend heavily on which school they attend
- The overall acceptance rate is an average across institutions with structurally different selection capacities

This notebook quantifies how much of the selection outcome is explained by institutional context versus individual characteristics, then fits a population-averaged model to estimate individual-level effects within that structure.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.model_acceptance import (
    load_data, engineer_student_features, engineer_military_features,
    engineer_ipeds_features, merge_ipeds, compute_icc,
    run_gee_logistic, gee_or_table,
    OUTCOME, STUDENT_COVARS, MILITARY_COVARS,
)

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.float_format', '{:.3f}'.format)


def sig_stars(p):
    """Convert p-value to significance stars."""
    if pd.isna(p): return ''
    return '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))


In [ ]:
# Load and engineer features
vsa_raw, ipeds_raw = load_data()

vsa   = engineer_student_features(vsa_raw)
vsa   = engineer_military_features(vsa) # adds mil_affil
ipeds = engineer_ipeds_features(ipeds_raw)
df    = merge_ipeds(vsa, ipeds)

print(f'Merged dataset (all):  {df.shape[0]:,} applicants, {df["Institution"].nunique()} institutions')


# Eligibility requirements
n_low_gpa = (df['GPA'] < 3.2).sum() 
n_noncitizen = (~df['us_citizen'].astype(bool)).sum()
n_non_partner = df['other_institution_flag'].sum()

df_eligible = df[
    (df['GPA'] >= 3.2) &
    (df['us_citizen'] == 1) &
    (~df['other_institution_flag'])
].copy()

# Count of filtered out for each eligibility reason
print(f'Excluded — GPA < 3.2: {n_low_gpa}')
print(f'Excluded — non-citizen: {n_noncitizen}')
print(f'Excluded — non-partner school: {n_non_partner}')
print(f'Eligible applicants: {df_eligible.shape[0]:,} across {df_eligible["Institution"].nunique()} partner institutions')

n_selected_all = df[~df['other_institution_flag']][OUTCOME].sum()
n_eligible     = df_eligible.shape[0]
print(f'Overall acceptance rate:        {n_selected_all / n_eligible:.1%}  '
      f'({int(n_selected_all)} selected / {n_eligible} eligible)')

## 1. Selection Rates Vary Dramatically by Institution

The overall rate masks enormous institution-level heterogeneity. Some institutions accept everyone they put forward; others accept nobody.

In [ ]:
# Acceptance rate per institution 
# Numerator : accepted at partner institutions (no GPA/citizenship filter)
# Denominator: eligible applicants only (GPA >= 3.2, US citizen, partner)

partner_df = df[~df['other_institution_flag']]
selected_by_inst = partner_df.groupby('Institution')[OUTCOME].sum().rename('n_selected')
eligible_by_inst = df_eligible.groupby('Institution').size().rename('n_eligible')

inst_stats = (
    pd.concat([selected_by_inst, eligible_by_inst], axis=1)
    .dropna(subset=['n_eligible'])
    .assign(acceptance_rate=lambda d: d['n_selected'] / d['n_eligible'])
    .sort_values('acceptance_rate')
    .reset_index()
)

print(f'Institution acceptance rates:')
print(f'  Min:    {inst_stats["acceptance_rate"].min():.0%}')
print(f'  Median: {inst_stats["acceptance_rate"].median():.0%}')
print(f'  Max:    {inst_stats["acceptance_rate"].max():.0%}')
print(f'  Std:    {inst_stats["acceptance_rate"].std():.0%}')
print(f'\nInstitutions with 0% acceptance:   {(inst_stats["acceptance_rate"] == 0).sum()}')
print(f'Institutions with 100% acceptance: {(inst_stats["acceptance_rate"] == 1).sum()}')
print(f'Institutions with >100%:           {(inst_stats["acceptance_rate"] > 1).sum()}'
      f'  <- accepted students who fall outside the eligibility window')


In [ ]:
# Institutions with >100% acceptance: accepted students outside the eligibility window
# These are students with GPA < 3.2 (or non-citizen) who were still selected.
# They appear in the numerator (partner-institution accepts) but not the denominator (eligible only).

over100 = inst_stats[inst_stats['acceptance_rate'] > 1].sort_values('acceptance_rate', ascending=False)
print(f'Institutions with acceptance rate > 100% (n={len(over100)}):')
print(over100[['Institution', 'n_selected', 'n_eligible', 'acceptance_rate']]
      .assign(acceptance_rate=lambda d: d['acceptance_rate'].map('{:.0%}'.format))
      .to_string(index=False))
print()

# Show the accepted-but-ineligible applicants at these schools
over100_insts = over100['Institution'].tolist()
accepted_ineligible = (
    df[df['Institution'].isin(over100_insts) & (df[OUTCOME] == 1)]
    .loc[lambda d: (d['GPA'] < 3.2) | (d['us_citizen'] != 1)]
    [['Institution', 'GPA', 'us_citizen', OUTCOME]]
    .sort_values('Institution')
)
print(f'Accepted applicants outside eligibility window at these institutions (n={len(accepted_ineligible)}):')
print(accepted_ineligible.to_string(index=False))
print()
print('Note: these applicants are excluded from df_eligible and all models below.')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: histogram of institution acceptance rates
axes[0].hist(inst_stats['acceptance_rate'], bins=20, edgecolor='white')
axes[0].axvline(inst_stats['acceptance_rate'].mean(), color='red',
                linestyle='--', linewidth=1.5, label=f'Mean ({inst_stats["acceptance_rate"].mean():.0%})')
axes[0].set_xlabel('Institution Acceptance Rate')
axes[0].set_ylabel('Number of Institutions')
axes[0].set_title('Distribution of Institution-Level Acceptance Rates')
axes[0].xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[0].legend()

# Right: caterpillar plot — acceptance rate per institution, sorted, dot-sized by n
ax = axes[1]
colors = inst_stats['acceptance_rate'].map(
    lambda r: '#d62728' if r == 0 else ('#2ca02c' if r == 1 else '#1f77b4')
)
ax.scatter(
    range(len(inst_stats)),
    inst_stats['acceptance_rate'],
    s=inst_stats['n_eligible'] * 3,
    c=colors, alpha=0.7, edgecolors='none'
)
ax.axhline(inst_stats['acceptance_rate'].mean(), color='red',
           linestyle='--', linewidth=1.2, label='Program mean')
ax.set_xlabel('Institution (ranked by acceptance rate)')
ax.set_ylabel('Acceptance Rate')
ax.set_title('Per-Institution Acceptance Rate (dot size = applicant n)\nRed = 0%, Green = 100%')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 2. ICC — Decomposing Selection Variance

The Intraclass Correlation Coefficient (ICC) quantifies what share of total outcome variance lies between institutions vs. within them.


In [ ]:
icc_result = compute_icc(df_eligible[OUTCOME], df_eligible['Institution'])

print('ICC Decomposition')
print('─' * 40)
print(f'  ICC (between-institution):  {icc_result["icc"]:.3f}')
print(f'  Between-institution var:    {icc_result["between_var"]:.4f}')
print(f'  Within-institution var:     {icc_result["within_var"]:.4f}')
print(f'  N institutions:             {icc_result["n_groups"]}')
print(f'  N observations:             {icc_result["n_obs"]:,}')
print()
pct = icc_result['icc'] * 100
print(f'Interpretation: {pct:.0f}% of selection variance is attributable to which'
      f' institution an applicant attends — not their individual characteristics.')

## 3. What This Means for Modeling

The ICC of ≈ 0.38 indicates that approximately 38% of the total variance in acceptance outcome is attributable to between-institution differences — differences in nomination culture, cohort composition, and selection thresholds that are not captured by individual applicant characteristics. The remaining ≈ 62% of variance is within-institution.

This partitioning motivates a multilevel modeling approach: GEE accounts for institutional clustering while estimating population-averaged effects on the within-institution component. 

Limitations:

- 41 clusters

- Approximately 25% of partner institutions contribute ≤ 5 eligible applicants

This suggests that coefficient estimates, particularly for predictors with limited within-institution variation, should be interpreted as preliminary and hypothesis-generating.

In [ ]:
# Cluster size distribution
cluster_sizes = df_eligible.groupby('Institution').size().sort_values()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram of cluster sizes
axes[0].hist(cluster_sizes, bins=range(1, cluster_sizes.max() + 2),
             edgecolor='white', align='left')
axes[0].axvline(cluster_sizes.median(), color='red', linestyle='--',
                label=f'Median = {cluster_sizes.median():.0f}')
axes[0].set_xlabel('Applicants per Institution')
axes[0].set_ylabel('Number of Institutions')
axes[0].set_title('Distribution of Institution Cluster Sizes')
axes[0].legend()

# Cumulative: what % of institutions have ≤ n applicants?
thresholds = range(1, 21)
cum_pcts = [(cluster_sizes <= t).mean() * 100 for t in thresholds]
axes[1].plot(thresholds, cum_pcts, marker='o', markersize=4)
axes[1].axhline(50, color='red', linestyle='--', linewidth=0.8, label='50%')
axes[1].set_xlabel('Applicants per Institution (threshold)')
axes[1].set_ylabel('% of Institutions')
axes[1].set_title('Cumulative % of Institutions with ≤ n Applicants')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter())
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Institutions with ≤5 applicants:   {(cluster_sizes <= 5).sum()} '
      f'({(cluster_sizes <= 5).mean():.0%})')
print(f'Institutions with ≤10 applicants:  {(cluster_sizes <= 10).sum()} '
      f'({(cluster_sizes <= 10).mean():.0%})')
print(f'Median cluster size: {cluster_sizes.median():.0f}')
print(f'Max cluster size:    {cluster_sizes.max()}')

In [ ]:
# Variance explained check: does GPA predict selection?
# Compare: (a) pooled, ignoring institution vs. (b) within-institution

analysis = df_eligible[['GPA', OUTCOME, 'Institution']].dropna().copy()

# Pooled correlation
r_pooled = analysis['GPA'].corr(analysis[OUTCOME])

# Within-institution: partial out institution means
analysis['GPA_demean']      = analysis['GPA'] - analysis.groupby('Institution')['GPA'].transform('mean')
analysis['outcome_demean']  = analysis[OUTCOME] - analysis.groupby('Institution')[OUTCOME].transform('mean')
r_within = analysis['GPA_demean'].corr(analysis['outcome_demean'])

print('GPA ↔ Selection correlation:')
print(f'  Pooled (ignores clustering):      r = {r_pooled:+.3f}')
print(f'  Within-institution (demeaned):    r = {r_within:+.3f}')

## 4. GEE Bivariate Screening

**Odds ratio (OR) from logistic regression** — quantifies how the odds of acceptance change per unit of each predictor, holding institution membership constant via GEE clustering.

**Bivariate screening**: fit one GEE model per predictor as a diagnostic step to see which predictors have any signal before entering a joint model.

**Known limitations going in:**
- `pell` has NaN for 'Prefer not to say' → those rows are dropped in any model
  that includes `pell`, reducing effective n
- GPA is range-restricted (≥ 3.2 by eligibility definition)
- With 75% acceptance rate and median n=18, there are few rejections to explain; CIs will be wide
- 11 of 41 institutions have 100% acceptance — no within-cluster variation to
  contribute to estimation for those schools

In [ ]:
# Bivariate GEE screening — one predictor at a time
all_covars = STUDENT_COVARS + MILITARY_COVARS  # 9 predictors

rows = []
for covar in all_covars:
    try:
        res  = run_gee_logistic(df_eligible, OUTCOME, [covar])
        tbl  = gee_or_table(res)
        row  = tbl.loc[covar]
        n    = int(df_eligible[[covar, OUTCOME]].dropna().shape[0])
        rows.append({'predictor': covar, 'n': n,
                     'OR': row['OR'], 'CI_lower': row['CI_lower'],
                     'CI_upper': row['CI_upper'], 'p_value': row['p_value']})
    except Exception as e:
        print(f'  WARNING: {covar} failed — {e}')
        rows.append({'predictor': covar, 'n': None,
                     'OR': None, 'CI_lower': None, 'CI_upper': None,
                     'p_value': None})

bivariate_tbl = pd.DataFrame(rows).set_index('predictor')
bivariate_tbl['CI_95'] = (
    bivariate_tbl['CI_lower'].map('{:.3f}'.format) + ' – ' +
    bivariate_tbl['CI_upper'].map('{:.3f}'.format)
)
bivariate_tbl['sig'] = bivariate_tbl['p_value'].apply(sig_stars)
print('Bivariate GEE Odds Ratios (clustered by Institution)')
print('─' * 60)
display(bivariate_tbl[['n', 'OR', 'CI_95', 'p_value', 'sig']].round(3))
print('\nNote: OR = 1.0 means no association. CIs crossing 1.0 are non-significant.')

## 5. Joint GEE Model — All Student-Level Predictors

Enter all predictors simultaneously. This controls for confounding between predictors.

**Limitation — 41 clusters for GEE.**  
With only 41 partner institutions, the SE estimator is biased downward — SEs are too small, CIs too narrow, and p-values anti-conservative.

Results should be treated as exploratory. The one statistically significant result (`mil_affil`) is the most suspect and is tested by a sensitivity check in Section 5.2 below.


In [ ]:
# Joint GEE: STUDENT_COVARS + MILITARY_COVARS
# mil_affil and rotc_air_force/rotc_army are mutually exclusive specifications;
# use mil_affil (any ROTC/cadet affiliation) for this first joint model.

joint_covars = STUDENT_COVARS + MILITARY_COVARS

res_joint = run_gee_logistic(df_eligible, OUTCOME, joint_covars)
tbl_joint = gee_or_table(res_joint)

tbl_joint['CI_95'] = (
    tbl_joint['CI_lower'].map('{:.3f}'.format) + ' – ' +
    tbl_joint['CI_upper'].map('{:.3f}'.format)
)
tbl_joint['sig'] = tbl_joint['p_value'].apply(sig_stars)

n_joint = int(df_eligible[joint_covars + [OUTCOME]].dropna().shape[0])
print(f'Joint GEE — n = {n_joint} (complete cases across all predictors)')
print(f'QIC: {res_joint.qic()[0]:.2f}')
print('─' * 60)
display(tbl_joint[['OR', 'CI_95', 'p_value', 'sig']].round(3))
print('\nNote: ORs are population-averaged (GEE), clustered by Institution.')
print('GPA is mean-centered within the eligible pool.')


**Note on mil_affil significance:** `mil_affil` was non-significant in bivariate screening 
(OR = 0.50, p = 0.073) but reaches significance in the joint model (OR = 0.42, p = 0.022). 
This is expected: controlling for GPA, demographics, and DoD scholar status removes confounding that was partially masking the military-affiliation effect — the joint model isolates the mil_affil signal by holding the other predictors constant.

### 5.1 Sensitivity Check — Independence Working Correlation

The exchangeable working correlation assumes all pairs of students within an institution are equally correlated. 

Re-fitting with an independence working correlation (treats observations as uncorrelated within clusters, relying entirely on the sandwich SE for correction) is a robustness check to check for changes.

In [ ]:
# Sensitivity: refit joint model with independence working correlation
res_indep = run_gee_logistic(
    df_eligible, OUTCOME, joint_covars, cov_struct='independence'
)
tbl_indep = gee_or_table(res_indep)
tbl_indep['sig'] = tbl_indep['p_value'].apply(sig_stars)

# Side-by-side comparison of key coefficients
comparison = pd.DataFrame({
    'OR_exch'  : tbl_joint['OR'],
    'p_exch'   : tbl_joint['p_value'],
    'sig_exch' : tbl_joint['sig'],
    'OR_indep' : tbl_indep['OR'],
    'p_indep'  : tbl_indep['p_value'],
    'sig_indep': tbl_indep['sig'],
}).drop(index='const')

print('Sensitivity Check — Exchangeable vs Independence Working Correlation')
print('─' * 70)
display(comparison.round(3))
print()

# Flag any predictors where significance changes
sig_changed = comparison[
    comparison['sig_exch'] != comparison['sig_indep']
]
if len(sig_changed):
    print(f'Predictors where significance changes across specifications:')
    display(sig_changed[['OR_exch','sig_exch','OR_indep','sig_indep']].round(3))
else:
    print('No predictors change significance across specifications.')
print()


### 5.2 Unpacking the mil_affil Coefficient

The GEE returned OR = 0.42 for `mil_affil` — the only significant predictor.
Now we will explore:

1. Is the gap due to GPA differences? (If mil students have lower GPAs they may be weaker candidates)
2. Is the gap within institutions or is it a composition effect? (Mil students may be concentrated at low-acceptance schools)
3. Is it uniform across ROTC branches or driven by a specific branch?
4. Which institutions are driving it?


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1: Acceptance by ROTC branch 
branch_map = {
    'Air Force': df_eligible[df_eligible['rotc_air_force']==1][OUTCOME].mean(),
    'Army'     : df_eligible[df_eligible['rotc_army']==1][OUTCOME].mean(),
    'Navy'     : df_eligible[(df_eligible['ROTCBranch']=='Navy')][OUTCOME].mean(),
    'No ROTC'  : df_eligible[df_eligible['mil_affil']==0][OUTCOME].mean(),
}
ns = {
    'Air Force': (df_eligible['rotc_air_force']==1).sum(),
    'Army'     : (df_eligible['rotc_army']==1).sum(),
    'Navy'     : (df_eligible['ROTCBranch']=='Navy').sum(),
    'No ROTC'  : (df_eligible['mil_affil']==0).sum(),
}
colors = ['#1f77b4','#d62728','#8c564b','#7f7f7f']
bars = axes[0].bar(branch_map.keys(), branch_map.values(), color=colors)
for bar, (k, n) in zip(bars, ns.items()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'n={n}', ha='center', va='bottom', fontsize=9)
axes[0].axhline(df_eligible[OUTCOME].mean(), color='black', linestyle='--',
                linewidth=1, label=f'Program mean ({df_eligible[OUTCOME].mean():.0%})')
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[0].set_title('Acceptance Rate by ROTC Affiliation')
axes[0].set_ylim(0, 1.1)
axes[0].legend(fontsize=8)

# Panel 2: Within-institution scatter (mil vs nonmil acceptance)
within_agg = (
    df_eligible.groupby(['Institution', 'mil_affil'])[OUTCOME]
    .agg(rate='mean', n='count')
    .unstack('mil_affil')
)
within_agg.columns = ['_'.join(str(c) for c in col) for col in within_agg.columns]
wdf = (
    within_agg.dropna()  # keep only institutions with both groups
    .rename(columns={'rate_0': 'nonmil', 'rate_1': 'mil',
                     'n_0': 'n_nonmil', 'n_1': 'n_mil'})
    .reset_index()
    .rename(columns={'Institution': 'inst'})
)

axes[1].scatter(wdf['nonmil'], wdf['mil'], s=wdf['n_mil']*12,
                alpha=0.7, color='#1f77b4', edgecolors='white')
axes[1].plot([0,1],[0,1], 'k--', linewidth=0.8, label='parity line')
# Label outliers (largest gap or largest n)
MIN_MIL_LABEL = 4  # label institutions with at least this many mil applicants
for _, row in wdf[wdf['n_mil'] >= MIN_MIL_LABEL].iterrows():
    short = row['inst'].replace('University','Univ.').replace('Institute','Inst.')
    short = short[:25]
    axes[1].annotate(short, (row['nonmil'], row['mil']),
                     fontsize=7, xytext=(4, 2), textcoords='offset points')
axes[1].set_xlabel('Non-mil acceptance rate')
axes[1].set_ylabel('Mil acceptance rate')
axes[1].set_title('Within-Institution: Mil vs Non-mil\n(dot size = n mil students)')
axes[1].xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[1].legend(fontsize=8)

# Panel 3: GPA distribution mil vs nonmil 
mil_gpa   = df_eligible[df_eligible['mil_affil']==1]['GPA'].dropna()
nonmil_gpa= df_eligible[df_eligible['mil_affil']==0]['GPA'].dropna()
axes[2].hist(nonmil_gpa, bins=20, alpha=0.6, density=True,
             label=f'Non-mil (n={len(nonmil_gpa)}, μ={nonmil_gpa.mean():.2f})', color='#7f7f7f')
axes[2].hist(mil_gpa, bins=20, alpha=0.7, density=True,
             label=f'ROTC/Cadet (n={len(mil_gpa)}, μ={mil_gpa.mean():.2f})', color='#1f77b4')
axes[2].set_xlabel('GPA')
axes[2].set_ylabel('Density')
axes[2].set_title('GPA Distribution: Mil vs Non-mil\n(eligibility floor = 3.2)')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

# Summary table
print('Within-institution summary (institutions with ≥1 mil and ≥1 nonmil eligible):')
print(f'  N institutions:              {len(wdf)}')
print(f'  Mil acceptance < nonmil:     {(wdf["mil"] < wdf["nonmil"]).sum()}')
print(f'  Mil acceptance > nonmil:     {(wdf["mil"] > wdf["nonmil"]).sum()}')
print(f'  Equal:                       {(wdf["mil"] == wdf["nonmil"]).sum()}')
print(f'  Avg within-inst gap (mil–nonmil): {(wdf["mil"] - wdf["nonmil"]).mean():.3f}')
print()
print('Acceptance by ROTC branch:')
for k, v in branch_map.items():
    print(f'  {k:12s}: {v:.1%}  (n={ns[k]})')


**Interpretation:**

- GPA distributions are nearly identical. Rule out academic preparation as the driver.
- Air Force (69%) is close to the program mean; Army (50%) and Navy (40%) are substantially below. The `mil_affil` coefficient in the GEE is primarily an Army/Navy ROTC effect.
- Difference is concentrated: **Northeastern** (10 mil students, 0% accepted) and **Virginia Tech** (10 mil, 10% vs 70% for nonmil) account for the majority of the gap, suggesting institution-specific selection criteria.

## 6. Summary: Why Individual-Level Modeling is Limited

Three structural facts shape what modeling VICEROY acceptance cannot inform:

| Constraint | Value | Implication |
|---|---|---|
| **ICC ≈ 0.38** | 38% of outcome variance is between partner institutions | No student-level covariate can explain this portion; institution context dominates |
| **Median cluster n ≈ 18** | Workable for most schools, but ~25% of partner institutions have ≤5 eligible applicants | Within-institution inference is feasible for larger schools but unreliable for the smallest clusters |
| **Selection mechanism is institutional capacity** | Partner institutions nominate a roughly fixed share of their pool regardless of individual characteristics | Student features explain at most the within-institution portion of variance (~60%) |

- Since the selection decision is primarily institutional, the path forward is to estimate individual effects of a few high volume-application institutions, or to pivot to a more interesting question. 
---

## 7. Next Steps

**Among selected scholars, who is selected for and actually completes an internship placement?**

That outcome:
- Varies at the individual level (not pre-determined by institutional capacity)
- Is plausibly influenced by academic preparation, military affiliation, discipline, and network
- Would allow the program to identify scholars most likely to convert to full pipeline
  participation

Current status: Waiting to receive access to internship placement data.